## E1.1 Clasificar campos como Fact o Dimension

In [0]:
%sql

SELECT COUNT(*) as count FROM bootcamp.silver.propiedades;

SELECT
    moneda
FROM bootcamp.silver.propiedades
GROUP BY moneda;

In [0]:
%sql


    SELECT
        COUNT(*) AS registros,    
        COUNT(DISTINCT p.partido) AS cardinalidad_partido,
        COUNT(DISTINCT p.tipo_operacion) AS cardinalidad_tipo_operacion,
        COUNT(DISTINCT p.estado) AS cardinalidad_estado,    
        COUNT(DISTINCT p.moneda) AS cardinalidad_moneda,
        COUNT(DISTINCT p.precio) AS cardinalidad_precio,
        COUNT(DISTINCT p.metros_cuadrados_totales) AS cardinalidad_m2_totales,
        COUNT(DISTINCT p.url) AS cardinalidad_url
    FROM bootcamp.silver.propiedades p





## E1.5 - SCD Type 1 — Overwrite con MERGE

In [0]:
%sql
SELECT COUNT(*) as prev_count FROM bootcamp.gold.dim_zona;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW staging AS SELECT * FROM bootcamp.gold.dim_zona;

    

In [0]:
%sql

SELECT * FROM staging;

In [0]:
%sql

SELECT region FROM staging GROUP BY region;

In [0]:
%sql

MERGE INTO staging AS target
USING bootcamp.gold.dim_zona AS source
ON target.zona_id = source.zona_id AND source.region ="gba zona norte"
WHEN MATCHED THEN UPDATE SET 
    target.ciudad="Gran Buenos Aires";


In [0]:
%sql

SELECT * FROM staging;

In [0]:
%sql

SELECT ciudad FROM staging GROUP BY ciudad;

In [0]:
%sql

MERGE INTO bootcamp.gold.dim_zona as t
USING staging as s
ON t.zona_id=s.zona_id
WHEN MATCHED THEN UPDATE SET
    t.ciudad=s.ciudad;

In [0]:
%sql

SELECT COUNT(*) as count FROM bootcamp.gold.dim_zona;

## E1.6 - SCD Type 2 — Crear tabla con columnas de historización

In [0]:
%sql

DROP TABLE IF EXISTS bootcamp.gold.dim_zona_scd2;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd2
(
    zona_id BIGINT NOT NULL GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1) COMMENT "PK",    
    partido STRING NOT NULL COMMENT "Partido donde se encuentra la propiedad",
    region STRING NOT NULL COMMENT "Region donde se encuentra la propiedad",
    ciudad STRING NOT NULL COMMENT "Ciudad donde se encuentra la propiedad",
    provincia STRING NOT NULL COMMENT "Provincia donde se encuentra la propiedad" DEFAULT "Buenos Aires",
    pais string NOT NULL COMMENT "Pais donde se encuentra la propiedad" DEFAULT "Argentina",
    valid_from TIMESTAMP NOT NULL COMMENT "Fecha desde la cual es valida la informacion",
    valid_to TIMESTAMP NOT NULL DEFAULT "9999-12-31" COMMENT "Fecha hasta la cual es valida la informacion",
    is_current BOOLEAN NOT NULL DEFAULT true COMMENT "Indica si la informacion es actual o no",    
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY(zona_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona SCD Type 2";


In [0]:
%sql

INSERT OVERWRITE bootcamp.gold.dim_zona_scd2 (    
    partido,
    region,
    ciudad,
    provincia,
    pais,
    valid_from
)
SELECT 
        DISTINCT 
        partido, 
        region,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'GBA'
        END as ciudad,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'Buenos Aires'
        END as provincia,
        'Argentina' as pais,
        CURRENT_TIMESTAMP() as valid_from
       
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL AND region IS NOT NULL
ORDER BY partido, region;
    

In [0]:
%sql

SELECT
    COUNT(*) as reg_is_current_false
FROM bootcamp.gold.dim_zona_scd2
WHERE is_current = false;

## E1.7 - SCD Type 2 — Historizar un cambio (manual)

In [0]:
%sql
UPDATE bootcamp.gold.dim_zona_scd2
    SET valid_to = CURRENT_TIMESTAMP(),
    is_current = FALSE
WHERE partido = 'capital federal'
AND is_current = TRUE;

In [0]:
%sql
INSERT INTO bootcamp.gold.dim_zona_scd2
(partido, region, ciudad,
valid_from)
VALUES (
'capital federal',
'capital federal',
'CABA',
CURRENT_TIMESTAMP()
);


In [0]:
%sql

SELECT * FROM bootcamp.gold.dim_zona_scd2 WHERE partido='capital federal' ORDER BY valid_from;

## E1.8 - SCD Type 2 — Historizar con MERGE (masivo)

In [0]:
%sql
SELECT DISTINCT partido FROM bootcamp.silver.propiedades ORDER BY partido;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW staging_zona_scd2 AS
SELECT * FROM VALUES
("avellaneda","gba zona sur", "CABA Sur"),
("barrio-nuevo","capital federal","CABA")
AS t(partido, region, ciudad);

In [0]:
SELECT * from staging_zona_scd2;

In [0]:
SELECT
    dest.*,
    src.*
FROM bootcamp.gold.dim_zona_scd2 dest
LEFT JOIN staging_zona_scd2 src
ON dest.partido==src.partido AND dest.region==src.region AND dest.is_current==true;

In [0]:
%sql

MERGE INTO bootcamp.gold.dim_zona_scd2 dest
USING staging_zona_scd2 src
ON dest.partido=src.partido AND dest.region=src.region AND dest.is_current=true
WHEN MATCHED AND dest.ciudad <> src.ciudad THEN 
UPDATE SET valid_to=CURRENT_TIMESTAMP(), is_current=FALSE
WHEN NOT MATCHED THEN
INSERT (partido, region, ciudad, valid_from)
VALUES (src.partido, src.region, src.ciudad, CURRENT_TIMESTAMP());

In [0]:
SELECT 
    1,
    src.*
FROM bootcamp.gold.dim_zona_scd2 dest
LEFT JOIN staging_zona_scd2 src
WHERE dest.partido==src.partido AND dest.region==src.region AND dest.is_current==true;

-- SELECT 1 FROM staging_zona_scd2;
-- SELECT 1 
--     FROM bootcamp.gold.dim_zona_scd2  s
--     WHERE partido = s.partido 
--       AND region = s.region 
--       AND is_current = false;

In [0]:
INSERT INTO bootcamp.gold.dim_zona_scd2
(partido, region, ciudad, valid_from)
SELECT     
    src.partido,
    src.region,
    src.ciudad,    
    CURRENT_TIMESTAMP() AS valid_from
FROM staging_zona_scd2 src
WHERE NOT EXISTS (
    SELECT 1 
    FROM bootcamp.gold.dim_zona_scd2 z
    WHERE z.partido == src.partido 
      AND z.region == src.region 
      AND z.is_current == true
);

In [0]:
%sql
select * from bootcamp.gold.dim_zona_scd2 where partido like 'avellaneda' OR partido like 'barrio-nuevo';

## E1.9 SCD Type 3 - Guardar valor anterior con MERGE

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_zona_scd3;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_zona_scd3 (
  zona_id BIGINT GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1),
  partido string NOT NULL,
  region string NOT NULL,
  ciudad string NOT NULL,
  provincia string NOT NULL,
  pais string NOT NULL,
  ciudad_anterior string NOT NULL,
  fecha_ultima_actualizacion TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY(zona_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Zona SCD Type 3";

In [0]:
%sql
INSERT OVERWRITE bootcamp.gold.dim_zona_scd3 (    
    partido,
    region,
    ciudad,
    provincia,
    pais,
    ciudad_anterior
)
SELECT 
        DISTINCT 
        partido, 
        region,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'GBA'
        END as ciudad,
        CASE
            WHEN region ='capital federal' THEN 'CABA'
            ELSE 'Buenos Aires'
        END as provincia,
        'Argentina' as pais,        
        "" as ciudad_anterior
FROM bootcamp.silver.propiedades
WHERE partido IS NOT NULL AND region IS NOT NULL
ORDER BY partido, region;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW staging_zona_scd3 AS
SELECT * FROM VALUES
("avellaneda","gba zona sur", "CABA Sur"),
("barrio-nuevo","capital federal","CABA")
AS t(partido, region, ciudad);

In [0]:
select * from staging_zona_scd3;

In [0]:
merge into bootcamp.gold.dim_zona_scd3 as dest
using staging_zona_scd3 as src
on dest.partido = src.partido and dest.region = src.region
when matched then update set 
dest.ciudad_anterior = dest.ciudad,
dest.ciudad=src.ciudad,
dest.fecha_ultima_actualizacion=CURRENT_TIMESTAMP()
when not matched then insert 
(partido, region, ciudad, provincia, pais, ciudad_anterior)
values (
    src.partido, 
    src.region, 
    src.ciudad, 
    CASE
        WHEN region ='capital federal' THEN 'CABA'
        ELSE 'Buenos Aires'
    END,
    'Argentina', 
    ""
    );


In [0]:
%sql

select * from bootcamp.gold.dim_zona_scd3 where ciudad_anterior <> '';

## E2.2 dim_tipo_operacion + dim_caracteristicas — DDL + ETL

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_tipo_operacion;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_tipo_operacion(
    tipo_operacion_id BIGINT GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1),
    tipo_operacion string NOT NULL,   
    categoria string NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Tipo Operacion SCD1";

In [0]:
DROP TABLE IF EXISTS bootcamp.gold.dim_caracteristicas;

CREATE TABLE IF NOT EXISTS bootcamp.gold.dim_caracteristicas(
    caracteristicas_id BIGINT GENERATED ALWAYS AS IDENTITY (INCREMENT BY 1 START WITH 1),
    estado string NOT NULL,
    cochera boolean NOT NULL,
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Dimension Caracteristicas SCD1";

In [0]:
%sql

select distinct tipo_operacion from bootcamp.silver.propiedades;

In [0]:
%sql

select estado, count(*) as count from bootcamp.silver.propiedades where estado is not null group by estado order by count desc;

select distinct cochera  from bootcamp.silver.propiedades;

In [0]:
%sql

insert overwrite bootcamp.gold.dim_tipo_operacion
(
    tipo_operacion,
    categoria
)
select 
    distinct tipo_operacion,
    case 
        when tipo_operacion in('venta','alquiler') then 'residencial'
        when tipo_operacion = 'temporal' then 'temporal'
        else 'Otros'
    end as categoria
from 
    bootcamp.silver.propiedades;

In [0]:
%sql

insert overwrite bootcamp.gold.dim_caracteristicas
(
    estado,
    cochera
)
select
    distinct estado,
    cochera
from bootcamp.silver.propiedades;


## E2.4 - Verificar todas las dimensiones

In [0]:
%sql

select 
    'dim_caracteristicas' as dim,
    count(*) as registros,
    registros - count(estado) as registros_nulos
from bootcamp.gold.dim_caracteristicas

union all

select 
    'dim_orientacion' as dim,
    count(*) as registros,
    registros - count(orientacion) as registros_nulos
from bootcamp.gold.dim_orientacion

union all

select 
    'dim_tiempo' as dim,
    count(*) as registros,
    registros - count(fecha) as registros_nulos
from bootcamp.gold.dim_tiempo

union all

select 
    'dim_tipo_operacion' as dim,
    count(*) as registros,
    registros - count(tipo_operacion) as registros_nulos
from bootcamp.gold.dim_tipo_operacion

union all

select 
    'dim_zona' as dim,
    count(*) as registros,
    registros - count(partido) as registros_nulos   
from bootcamp.gold.dim_zona;

## E2.5 fact_propiedades - DDL con row_hash

In [0]:
%sql
DROP TABLE IF EXISTS bootcamp.gold.fact_propiedades;

CREATE TABLE IF NOT EXISTS bootcamp.gold.fact_propiedades(
    row_hash STRING NOT NULL COMMENT "PK-ROW HASH único de la propiedad",
    zona_id BIGINT NOT NULL COMMENT "FK - dim_zona",
    tipo_orientacion_id BIGINT NOT NULL COMMENT "FK-dim_orientacion",
    fecha_id BIGINT NOT NULL COMMENT "FK-dim_tiempo",
    caracteristicas_id BIGINT NOT NULL COMMENT "FK-dim_caracteristicas",
    tipo_operacion_id BIGINT NOT NULL COMMENT "FK-dim_tipo_operacion",
    url STRING NOT NULL COMMENT "URL de la propiedad",
    precio DECIMAL(15,2),
    expensas DECIMAL(15,2),
    precio_por_m2 DECIMAL(15,2),
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    ambientes INT,    
    _createdAt TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP() COMMENT 'DateTime in UTC',
    PRIMARY KEY(row_hash),   
    FOREIGN KEY (zona_id)
    REFERENCES bootcamp.gold.dim_zona (zona_id),
    FOREIGN KEY (tipo_orientacion_id)
    REFERENCES bootcamp.gold.dim_orientacion (orientacion_id),
    FOREIGN KEY (fecha_id)
    REFERENCES bootcamp.gold.dim_tiempo (fecha_id),
    FOREIGN KEY (caracteristicas_id)
    REFERENCES bootcamp.gold.dim_caracteristicas (caracteristicas_id),
    FOREIGN KEY (tipo_operacion_id)
    REFERENCES bootcamp.gold.dim_tipo_operacion (tipo_operacion_id)
)
USING DELTA
TBLPROPERTIES(
   delta.enableChangeDataFeed = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact = true,
   delta.feature.allowColumnDefaults='supported'
)
COMMENT "Fact Propiedades";

## E2.6 ETL - LEFT JOINs

In [0]:
%sql
SELECT
    'silver.propiedades' as tabla,
    count(*) as count
from bootcamp.silver.propiedades

UNION ALL

SELECT
    'gold.fact_propiedades' as tabla,
    COUNT(*) as count
from bootcamp.gold.fact_propiedades;

In [0]:
%sql
WITH gold as (
    SELECT 
            MD5(CONCAT_WS('|', p.url, CAST(p.precio AS STRING))) as row_hash,
            z.zona_id,
            ori.orientacion_id as tipo_orientacion_id,
            t.fecha_id,
            c.caracteristicas_id,
            op.tipo_operacion_id,
            p.url,
            p.precio,
            p.expensas,
            p.precio_por_m2,
            p.metros_cuadrados_totales,
            p.metros_cuadrados_cubiertos,
            p.ambientes
    from bootcamp.silver.propiedades as p
    left join bootcamp.gold.dim_zona as z on p.partido==z.partido and p.region == z.region
    left join bootcamp.gold.dim_orientacion as ori on p.orientacion==ori.orientacion
    left join bootcamp.gold.dim_tiempo as t on p.fecha_publicacion==t.fecha
    left join bootcamp.gold.dim_caracteristicas as c on p.estado == c.estado and p.cochera == c.cochera
    left join bootcamp.gold.dim_tipo_operacion as op on p.tipo_operacion == op.tipo_operacion
)

merge into bootcamp.gold.fact_propiedades as dest
using gold as src
ON dest.row_hash = src.row_hash
when not matched then insert
(
    row_hash,
    zona_id,
    tipo_orientacion_id,
    fecha_id,
    caracteristicas_id,
    tipo_operacion_id,
    url,
    precio,
    expensas,
    precio_por_m2,
    metros_cuadrados_totales,
    metros_cuadrados_cubiertos,
    ambientes
)
values
(
    src.row_hash,
    src.zona_id,
    src.tipo_orientacion_id,
    src.fecha_id,
    src.caracteristicas_id,
    src.tipo_operacion_id,
    src.url,
    src.precio,
    src.expensas,
    src.precio_por_m2,
    src.metros_cuadrados_totales,
    src.metros_cuadrados_cubiertos,
    src.ambientes
);




In [0]:
%sql
SELECT
    'silver.propiedades' as tabla,
    count(*) as count
from bootcamp.silver.propiedades

UNION ALL

SELECT
    'gold.fact_propiedades' as tabla,
    COUNT(*) as count
from bootcamp.gold.fact_propiedades;

In [0]:
%sql

SELECT
    COUNT(*) as registros,
    registros - count(zona_id) as zona_nulls,
    registros - count(tipo_orientacion_id) as orientacion_null,
    registros - count(fecha_id) as fecha_nulls,
    registros - count(tipo_operacion_id) as tipo_op_nulls
from bootcamp.gold.fact_propiedades;

## E3.1 - Top 5 partidos por precio  promedio m2

In [0]:
%sql
select
    z.partido,
    op.tipo_operacion,
    count(*) as  total_propiedades,
    avg(p.precio_por_m2) as avg_precio_m2
from bootcamp.gold.fact_propiedades as p
inner join bootcamp.gold.dim_zona as z on p.zona_id = z.zona_id
inner join bootcamp.gold.dim_tipo_operacion op on p.tipo_operacion_id = op.tipo_operacion_id
group by z.partido, op.tipo_operacion
order by avg_precio_m2 desc;




## E3.2 Distribucion por tipo de operacion y categoria

In [0]:
%sql

SELECT
    op.tipo_operacion,
    op.categoria,
    count(*) as total_propiedades,
    median(f.precio) as precio_medio,
    avg(f.precio) as precio_promedio
from bootcamp.gold.fact_propiedades as f
inner join bootcamp.gold.dim_tipo_operacion as op on f.tipo_operacion_id = op.tipo_operacion_id
group by op.tipo_operacion, op.categoria
order by total_propiedades desc;

## E3.3 - Evolucion mensual del precio promedio

In [0]:
%sql

select
    t.anio,
    t.mes,
    count(*) as total_propiedades,
    avg(f.precio) as precio_promedio
    --SUM(COUNT(*)) OVER() AS total_gral_propiedades
from bootcamp.gold.fact_propiedades as f
inner join bootcamp.gold.dim_tiempo as t on f.fecha_id = t.fecha_id
group by t.anio, t.mes
order by t.anio, t.mes;


## E3.4 cruce dimensiones

In [0]:
%sql

select
    z.partido,
    op.tipo_operacion,
    c.estado,
    c.cochera,
    count(*) as total_props,
    avg(f.precio) as precio_promedio
from bootcamp.gold.fact_propiedades as f
inner join bootcamp.gold.dim_zona as z on f.zona_id=z.zona_id
inner join bootcamp.gold.dim_tipo_operacion as op on f.tipo_operacion_id = op.tipo_operacion_id
inner join bootcamp.gold.dim_caracteristicas as c on c.caracteristicas_id = f.caracteristicas_id
where z.region="capital federal"
group by z.partido, op.tipo_operacion, c.estado, c.cochera
order by total_props desc;